# Fertilizer Recommendation Model
**Tea Plantation Smart Monitoring System**

This notebook trains:
1. **GBC classifier** — predicts fertilizer *type* from soil + weather inputs.
2. **Per-type Linear Regression models** — predicts application *quantity* (kg/ha) for each fertilizer type.

Features: `N`, `P`, `K`, `pH`, `plant_type`, `weather_condition`
Targets: `fertilizer_type` (classification), `quantity_kg_ha` (regression)

Artifacts: `fertilizer_type_gbc.pkl`, `fertilizer_amount_models.pkl`, `fertilizer_encoders.pkl`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, mean_absolute_error, r2_score)

ARTIFACTS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'ml_training', 'model_artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f'Artifacts will be saved to: {ARTIFACTS_DIR}')


## 1. Data Generation
Synthetic agronomic dataset. Replace the block marked `# REPLACE WITH REAL DATA` with a `pd.read_csv()` call.


In [ ]:
# REPLACE WITH REAL DATA
# To use real data: df = pd.read_csv('path/to/fertilizer_data.csv')
# Expected columns: N, P, K, pH, plant_type, weather_condition, fertilizer_type, quantity_kg_ha

np.random.seed(42)
N_SAMPLES = 2500

plant_types      = ['Young Tea', 'Mature Tea', 'Old Tea']
weather_conds    = ['Sunny', 'Cloudy', 'Rainy', 'Stormy']
fertilizer_types = ['Urea', 'NPK_20_10_10', 'NPK_17_17_17', 'Ammonium_Sulfate', 'Organic_Compost']

plant_col   = np.random.choice(plant_types,   size=N_SAMPLES)
weather_col = np.random.choice(weather_conds, size=N_SAMPLES)
fert_col    = np.random.choice(fertilizer_types, size=N_SAMPLES,
                               p=[0.25, 0.25, 0.20, 0.15, 0.15])

records = []
for i in range(N_SAMPLES):
    ft = fert_col[i]
    wc = weather_col[i]
    pt = plant_col[i]

    # Base NPK varies by fertilizer type
    base_N = {'Urea': 80, 'NPK_20_10_10': 50, 'NPK_17_17_17': 45,
              'Ammonium_Sulfate': 60, 'Organic_Compost': 20}[ft]
    base_P = {'Urea': 15, 'NPK_20_10_10': 30, 'NPK_17_17_17': 35,
              'Ammonium_Sulfate': 10, 'Organic_Compost': 25}[ft]
    base_K = {'Urea': 10, 'NPK_20_10_10': 25, 'NPK_17_17_17': 35,
              'Ammonium_Sulfate': 8, 'Organic_Compost': 30}[ft]
    base_pH = {'Urea': 5.5, 'NPK_20_10_10': 6.0, 'NPK_17_17_17': 6.2,
               'Ammonium_Sulfate': 4.8, 'Organic_Compost': 6.5}[ft]

    # Quantity influenced by plant type and weather
    qty_base = {'Urea': 120, 'NPK_20_10_10': 150, 'NPK_17_17_17': 140,
                'Ammonium_Sulfate': 110, 'Organic_Compost': 300}[ft]
    qty_plant_mod = {'Young Tea': -20, 'Mature Tea': 0, 'Old Tea': 15}[pt]
    qty_weather_mod = {'Sunny': 0, 'Cloudy': -5, 'Rainy': -15, 'Stormy': -25}[wc]

    records.append({
        'N':                  base_N + np.random.normal(0, 8),
        'P':                  base_P + np.random.normal(0, 5),
        'K':                  base_K + np.random.normal(0, 5),
        'pH':                 np.clip(base_pH + np.random.normal(0, 0.3), 3.5, 8.0),
        'plant_type':         pt,
        'weather_condition':  wc,
        'fertilizer_type':    ft,
        'quantity_kg_ha':     max(0, qty_base + qty_plant_mod + qty_weather_mod + np.random.normal(0, 15))
    })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
print(f'\nFertilizer type distribution:\n{df["fertilizer_type"].value_counts()}')
df.head()


## 2. Exploratory Data Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['fertilizer_type'].value_counts()
axes[0].barh(counts.index, counts.values, color=sns.color_palette('Set2', len(counts)))
axes[0].set_title('Fertilizer Type Distribution')
axes[0].set_xlabel('Count')

numeric_cols = ['N', 'P', 'K', 'pH', 'quantity_kg_ha']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1], square=True)
axes[1].set_title('Numeric Feature Correlation')
plt.tight_layout()
plt.show()

# Quantity by fertilizer type
fig, ax = plt.subplots(figsize=(12, 5))
df.boxplot(column='quantity_kg_ha', by='fertilizer_type', ax=ax)
ax.set_title('Quantity (kg/ha) by Fertilizer Type')
ax.set_xlabel('Fertilizer Type')
ax.set_ylabel('quantity_kg_ha')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('=== Summary Statistics ===')
display(df.describe())


## 3. Label Encoding
Encode `plant_type`, `weather_condition`, and `fertilizer_type`.


In [ ]:
le_plant   = LabelEncoder()
le_weather = LabelEncoder()
le_fert    = LabelEncoder()

df['plant_type_enc']      = le_plant.fit_transform(df['plant_type'])
df['weather_cond_enc']    = le_weather.fit_transform(df['weather_condition'])
df['fertilizer_type_enc'] = le_fert.fit_transform(df['fertilizer_type'])

print('plant_type classes:     ', list(le_plant.classes_))
print('weather_condition classes:', list(le_weather.classes_))
print('fertilizer_type classes:', list(le_fert.classes_))

encoders = {
    'le_plant':   le_plant,
    'le_weather': le_weather,
    'le_fert':    le_fert
}


## 4. GBC — Fertilizer Type Classification (80/20 split)


In [ ]:
clf_features = ['N', 'P', 'K', 'pH', 'plant_type_enc', 'weather_cond_enc']

X_clf = df[clf_features].values
y_clf = df['fertilizer_type_enc'].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

gbc = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                  learning_rate=0.1, random_state=42)
gbc.fit(X_train_c, y_train_c)

train_acc = accuracy_score(y_train_c, gbc.predict(X_train_c))
test_acc  = accuracy_score(y_test_c,  gbc.predict(X_test_c))
print(f'GBC Fertilizer Type  —  Train accuracy: {train_acc:.4f}  |  Test accuracy: {test_acc:.4f}')

# Feature importances
fi = pd.Series(gbc.feature_importances_, index=clf_features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
fi.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('GBC Feature Importances — Fertilizer Type')
plt.tight_layout()
plt.show()


## 5. Per-Type Linear Regression — Quantity (kg/ha)
A separate `LinearRegression` is trained for each fertilizer type.


In [ ]:
reg_features = ['N', 'P', 'K', 'pH', 'plant_type_enc', 'weather_cond_enc']

amount_models = {}   # {fertilizer_label: LinearRegression}
reg_results   = []

for enc_val, fert_name in enumerate(le_fert.classes_):
    subset = df[df['fertilizer_type_enc'] == enc_val]
    if len(subset) < 20:
        print(f'  Skipping {fert_name}: only {len(subset)} samples.')
        continue

    X_r = subset[reg_features].values
    y_r = subset['quantity_kg_ha'].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_r, y_r, test_size=0.2, random_state=42
    )

    lr = LinearRegression()
    lr.fit(X_tr, y_tr)

    y_pred_r = lr.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred_r)
    r2  = r2_score(y_te, y_pred_r)

    amount_models[fert_name] = lr
    reg_results.append({'Fertilizer Type': fert_name, 'Samples': len(subset),
                         'MAE (kg/ha)': mae, 'R²': r2})
    print(f'  {fert_name:<22}  n={len(subset):<5}  MAE={mae:.2f}  R²={r2:.4f}')

print('\n=== Regression Results Summary ===')
display(pd.DataFrame(reg_results))


## 6. Confusion Matrix (Fertilizer Type GBC)


In [ ]:
y_pred_c = gbc.predict(X_test_c)
cm = confusion_matrix(y_test_c, y_pred_c)
class_names = le_fert.classes_

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — Fertilizer Type GBC', fontsize=14)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\n=== Classification Report ===')
print(classification_report(y_test_c, y_pred_c, target_names=class_names))


## 7. Save Artifacts


In [ ]:
artifacts = {
    'fertilizer_type_gbc.pkl':    gbc,
    'fertilizer_amount_models.pkl': amount_models,
    'fertilizer_encoders.pkl':    encoders
}

for filename, obj in artifacts.items():
    path = os.path.join(ARTIFACTS_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved: {path}')

print('\nAll fertilizer model artifacts saved successfully.')
